# 02 — FinBERT document embeddings

Generate a fixed-size embedding per filing by running each cleaned EDGAR document through the fine-tuned encoder in `artifacts/finbert-mlm/`, taking the `[CLS]` of each 512-token chunk (64-token stride), and mean-pooling chunks within a document.

**Inputs**
- `artifacts/finbert-mlm/` — encoder + tokenizer (from `01_finbert_finetune.ipynb`)
- `data/interim/edgar_text/{cik}/{accession}.txt` — cleaned filings (from `src/data/clean_filings.py`)
- `data/processed/edgar_index.parquet` — accession-level metadata (gives `filing_date`, `form_type`)

**Output**
- `data/processed/finbert_doc_embed/year=YYYY/part-0.parquet` — rows `(cik, accession, filing_date, form_type, n_chunks, vec)` where `vec` is a 768-dim float32 list.

**Resumability:** sharded by `filing_date.year`. On re-run, any year-shard already on disk is skipped. To re-do a year, delete its `part-0.parquet`.

Pairs with `03_finbert_embed_stockday.ipynb`, which aggregates these per-filing vectors into a `(permno, date)` panel.

## A. Setup

In [ ]:
from __future__ import annotations
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import torch
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
MODEL_DIR = REPO_ROOT / 'artifacts' / 'finbert-mlm'
TEXT_DIR = REPO_ROOT / 'data' / 'interim' / 'edgar_text'
INDEX_PATH = REPO_ROOT / 'data' / 'processed' / 'edgar_index.parquet'
OUT_DIR = REPO_ROOT / 'data' / 'processed' / 'finbert_doc_embed'
SUMMARY_PATH = REPO_ROOT / 'reports' / 'metrics' / 'finbert_doc_embed_summary.json'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device={device}')
if device.type == 'cuda':
    print(f'gpu={torch.cuda.get_device_name(0)}, bf16={torch.cuda.is_bf16_supported()}')

## B. Load fine-tuned encoder

Loads as `AutoModel` (drops the MLM head — we only need the encoder), bf16 on GPU. All parameters frozen — pure inference.

In [ ]:
DTYPE = torch.bfloat16 if device.type == 'cuda' and torch.cuda.is_bf16_supported() else torch.float32
tok = AutoTokenizer.from_pretrained(MODEL_DIR)
mdl = AutoModel.from_pretrained(MODEL_DIR, torch_dtype=DTYPE).to(device).eval()
for p in mdl.parameters():
    p.requires_grad_(False)
HIDDEN = mdl.config.hidden_size
print(f'encoder ready: hidden={HIDDEN}, params={mdl.num_parameters()/1e6:.1f}M, dtype={DTYPE}')

## C. Build work queue

Join `edgar_index` to the cleaned-text directory. Filings whose cleaned `.txt` is missing (extraction failure, below min length, etc.) are dropped.

In [ ]:
idx = pd.read_parquet(INDEX_PATH)
idx['clean_path'] = [TEXT_DIR / cik / f'{acc}.txt' for cik, acc in zip(idx['cik'], idx['accession'])]
idx['exists'] = [p.exists() for p in idx['clean_path']]
work = idx[idx['exists']].copy()
work['year'] = work['filing_date'].dt.year
print(f'edgar_index rows: {len(idx):,}')
print(f'with cleaned text:  {len(work):,}')
print()
print('filings per year:')
print(work.groupby('year').size().to_frame('n'))

## D. Embedding function + dry run

`embed_text(text)` tokenizes with `(max_length=512, stride=64, return_overflowing_tokens=True)` — same chunking used during MLM training — then mean-pools the chunk `[CLS]` vectors to a single 768-dim float32.

The dry run validates shape, finiteness, and rough per-doc cost on 5 random filings before launching the full pass. **Gate: if `dt` per doc looks pathological or `vec` contains NaN/Inf, stop and debug before Section E.**

In [ ]:
CHUNK_LEN = 512
STRIDE = 64
CHUNK_BATCH = 16

@torch.no_grad()
def embed_text(text: str) -> tuple[np.ndarray, int]:
    enc = tok(
        text,
        max_length=CHUNK_LEN,
        truncation=True,
        stride=STRIDE,
        return_overflowing_tokens=True,
        return_tensors='pt',
        padding='max_length',
    )
    ids = enc['input_ids'].to(device)
    mask = enc['attention_mask'].to(device)
    n_chunks = int(ids.size(0))
    cls = []
    for i in range(0, n_chunks, CHUNK_BATCH):
        out = mdl(input_ids=ids[i:i+CHUNK_BATCH], attention_mask=mask[i:i+CHUNK_BATCH])
        cls.append(out.last_hidden_state[:, 0, :].float())
    vec = torch.cat(cls, dim=0).mean(dim=0).cpu().numpy().astype(np.float32)
    return vec, n_chunks

sample = work.sample(5, random_state=0)
for _, row in sample.iterrows():
    text = Path(row['clean_path']).read_text(encoding='utf-8', errors='ignore')
    t0 = time.perf_counter()
    vec, nc = embed_text(text)
    dt = time.perf_counter() - t0
    print(f"{row['cik']}/{row['accession']} form={row['form_type']:>4} chars={len(text):>7} chunks={nc:>3} norm={np.linalg.norm(vec):.2f} dt={dt:.2f}s")
print(f'\nvec shape: {vec.shape}, dtype: {vec.dtype}, any_nan: {np.isnan(vec).any()}, any_inf: {np.isinf(vec).any()}')

## E. Full inference (sharded, resumable)

Year-by-year. Each year writes atomically (tmp → rename) to `data/processed/finbert_doc_embed/year=YYYY/part-0.parquet`. Existing shards are skipped — delete a year's `part-0.parquet` to redo it. Empty/unreadable filings are logged and skipped without aborting the year.

The whole pass is GPU-bound. Wall-clock will be measured on first run — don't trust any a-priori estimate.

In [ ]:
SCHEMA = pa.schema([
    ('cik', pa.string()),
    ('accession', pa.string()),
    ('filing_date', pa.timestamp('ns')),
    ('form_type', pa.string()),
    ('n_chunks', pa.int32()),
    ('vec', pa.list_(pa.float32())),
])

def shard_path(year: int) -> Path:
    return OUT_DIR / f'year={year}' / 'part-0.parquet'

def process_year(year: int, group: pd.DataFrame) -> int:
    out_path = shard_path(year)
    if out_path.exists():
        n = pq.read_metadata(out_path).num_rows
        print(f'year={year}: exists ({n} rows) — skipping')
        return n
    out_path.parent.mkdir(parents=True, exist_ok=True)
    rows = []
    skipped = 0
    pbar = tqdm(group.itertuples(index=False), total=len(group), desc=f'year={year}')
    for r in pbar:
        try:
            text = Path(r.clean_path).read_text(encoding='utf-8', errors='ignore')
            if len(text.strip()) < 100:
                skipped += 1
                continue
            vec, nc = embed_text(text)
        except Exception as e:
            pbar.write(f'skip {r.cik}/{r.accession}: {type(e).__name__}: {e}')
            skipped += 1
            continue
        rows.append({
            'cik': r.cik,
            'accession': r.accession,
            'filing_date': pd.Timestamp(r.filing_date),
            'form_type': r.form_type,
            'n_chunks': nc,
            'vec': vec.tolist(),
        })
    if not rows:
        print(f'year={year}: no rows produced')
        return 0
    table = pa.Table.from_pylist(rows, schema=SCHEMA)
    tmp = out_path.with_suffix('.parquet.tmp')
    pq.write_table(table, tmp, compression='zstd')
    tmp.rename(out_path)
    print(f'year={year}: wrote {len(rows)} rows, skipped {skipped} -> {out_path}')
    return len(rows)

YEARS = sorted(int(y) for y in work['year'].unique())
print(f'years: {YEARS[0]}..{YEARS[-1]} ({len(YEARS)} years)\n')
for year in YEARS:
    grp = work[work['year'] == year]
    process_year(year, grp)

## F. Validation + summary

Verify total rowcount against the work-queue size, sample-check for NaN/Inf in embeddings, and write a summary JSON to `reports/metrics/`.

In [ ]:
import duckdb
con = duckdb.connect()
total = con.sql(f"SELECT COUNT(*) FROM '{OUT_DIR}/year=*/*.parquet'").fetchone()[0]
expected = int(len(work))
print(f'embedded filings: {total:,} / expected with cleaned text: {expected:,}')

sample_df = con.sql(f"SELECT vec FROM '{OUT_DIR}/year=*/*.parquet' USING SAMPLE 500 ROWS").df()
vecs = np.stack([np.asarray(v, dtype=np.float32) for v in sample_df['vec'].values])
print(f'sample shape: {vecs.shape}, has_nan: {np.isnan(vecs).any()}, has_inf: {np.isinf(vecs).any()}')
print(f'norm: mean={np.linalg.norm(vecs, axis=1).mean():.3f}, std={np.linalg.norm(vecs, axis=1).std():.3f}')

summary = {
    'total_embedded': int(total),
    'input_filings_with_text': expected,
    'years': YEARS,
    'hidden_dim': int(HIDDEN),
    'chunk_len': CHUNK_LEN,
    'chunk_stride': STRIDE,
    'model_dir': str(MODEL_DIR.relative_to(REPO_ROOT)),
    'output_dir': str(OUT_DIR.relative_to(REPO_ROOT)),
}
SUMMARY_PATH.parent.mkdir(parents=True, exist_ok=True)
SUMMARY_PATH.write_text(json.dumps(summary, indent=2))
print(f'summary -> {SUMMARY_PATH.relative_to(REPO_ROOT)}')